# Markdown Notes MCP server

This notebook deploys the custom Markdown Notes MCP server and uses its tools to search, save, and read Markdown notes stored on a PVC. See the [README](./README.md) for how the server works and how to build your own image.

MCP servers must be enabled on your prokube cluster (v1.9.0 or later).

## Deploy

In [ ]:
!kubectl apply -f markdown-notes.yaml
!kubectl wait --for=condition=Ready mcpservers.toolhive.stacklok.dev/markdown-notes-example --timeout=3m

## Connect from this Lab

Labs reach the workspace's MCP endpoint through the internal Agent Gateway URL without an API key. This endpoint combines all MCP servers in the workspace and prefixes each tool with its server name, for example `markdown-notes-example_search_notes`.

`McpSession` is a minimal MCP client from this repository. Any MCP client library can connect to the same URL.

In [ ]:
%run -n ../../src/pk_helpers/mcp.py

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

session = connect_when_ready(workspace_mcp_url(namespace), tool_prefix="markdown-notes-example_")
session.list_tools()

Search the two sample notes the server creates on first start:

In [ ]:
result = session.call_tool("markdown-notes-example_search_notes", {"query": "quota"})
result["structuredContent"]["result"]

Save a note and read it back. Note names are stored as lowercase slugs:

In [ ]:
session.call_tool(
    "markdown-notes-example_save_note",
    {"name": "My first note", "content": "# My first note\n\nSaved from a notebook."},
)
result = session.call_tool("markdown-notes-example_get_note", {"name": "my-first-note"})
print(result["content"][0]["text"])

List all notes:

In [ ]:
result = session.call_tool("markdown-notes-example_list_notes", {})
result["structuredContent"]["result"]

## Connect with an API key

Clients outside the cluster use the server's own route, `https://<your-prokube-domain>/svc/mcp/<workspace>/markdown-notes-example`, with an API key. Create a key with access to `markdown-notes-example` on the **API Keys** page at `https://<your-prokube-domain>/pkui/ai-gateway/keys`.

The next cell prompts for the key unless `MCP_API_KEY` is set and connects to the same route from inside the cluster. Tool names on this route have no prefix.

In [ ]:
%run -n ../../src/pk_helpers/api_key.py

api_key = get_or_create_mcp_api_key()
server_session = connect_when_ready(server_mcp_url(namespace, "markdown-notes-example"), api_key=api_key)
server_session.list_tools()

## Clean up

This also deletes the PVC with all notes.

In [ ]:
!kubectl delete -f markdown-notes.yaml